# SC4020 Project 2 — Task 3: Cold-Target Generalisation

**Problem.** The Task 2 CF predicts a target's score from that target's training scores. A *cold* target has none, so CF falls back to the ligand's mean score $\mu + b_u$ — the same value for every new target, so it cannot tell new targets apart.

**Idea (guilt-by-association).** Task 1 (Step 4c) showed that targets with similar amino-acid sequences have similar binding profiles. So for a cold target we borrow scores from the **training targets whose sequences are most similar**. This keeps the lecture 11.3 prediction formula (slides 23–24), but the neighbour similarity comes from the **sequence**, which a new target has, instead of from scores, which it does not have:

$$\hat r_{ui} = (\mu + b_u) + \frac{\sum_{j \in N_k(i;u)} s^{\text{seq}}_{ij}\,\big(r_{uj} - (\mu + b_u)\big)}{\sum_{j \in N_k(i;u)} s^{\text{seq}}_{ij}}$$

$N_k(i;u)$ = the $k$ training targets most similar in sequence to the cold target $i$ that ligand $u$ has a training score for. The target bias $b_i$ is unknown for a cold target, so the baseline part is the ligand mean $\mu + b_u$; the neighbours' deviations carry both their target bias and the ligand–target interaction.

**Sequence similarity: Jaccard on 3-mer sets (lecture 11.3).** Each protein is turned into the *set* of overlapping 3-letter fragments it contains (feature construction, lecture 12.2 slides 23–25), e.g. `MKKF...` → {`MKK`, `KKF`, ...}. Two targets are similar if they share many fragments:

$$s^{\text{seq}}_{ij} = J(A_i, A_j) = \frac{|A_i \cap A_j|}{|A_i \cup A_j|}$$

This is the Jaccard similarity from lecture 11.3, applied to sequence fragments instead of rated items.

As a **reference**, the same method is also run with the Smith–Waterman similarities supplied with the dataset (an alignment-based bioinformatics score), to see how close the simple lecture similarity gets.

Requires `data/clean/` from `task1_analysis.ipynb`.

In [ ]:
import json
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

SEED = 42
DATA_DIR = Path("data")
CLEAN_DIR = DATA_DIR / "clean"
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

pairs = {n: pd.read_csv(CLEAN_DIR / f"{n}_pairs.csv", dtype={"ligand_key": str, "target_key": str})
         for n in ["davis", "kiba"]}
targets = {n: pd.read_csv(CLEAN_DIR / f"{n}_targets.csv", dtype=str).set_index("target_key")
           for n in ["davis", "kiba"]}
{n: (len(p), len(targets[n])) for n, p in pairs.items()}

## Step 1: Cold-target split

Targets (not pairs) are split 70 / 15 / 15 with the shared seed. **All** pairs of a validation or test target are held out, so these targets have no training scores at all. Validation targets are used to choose $k$; test targets are evaluated once.

Because Task 1 merged targets with identical sequences, a cold target can never have an exact sequence copy in training (otherwise the task would be trivial).

In [ ]:
def cold_target_split(df, seed=SEED, fracs=(0.70, 0.15, 0.15)):
    tgts = np.array(sorted(df["target_key"].unique()))
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(tgts))
    n_tr, n_va = int(fracs[0] * len(tgts)), int(fracs[1] * len(tgts))
    role = {}
    for i, idx in enumerate(order):
        role[tgts[idx]] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
    df = df.assign(split=df["target_key"].map(role))
    # a ligand needs training scores (for mu + b_u); drop the rare evaluation pairs whose ligand has none
    train_ligs = set(df.loc[df["split"] == "train", "ligand_key"])
    dropped = int(((df["split"] != "train") & ~df["ligand_key"].isin(train_ligs)).sum())
    df = df[(df["split"] == "train") | df["ligand_key"].isin(train_ligs)]
    return df, role, dropped

splits, roles, rows = {}, {}, []
for name, df in pairs.items():
    splits[name], roles[name], dropped = cold_target_split(df)
    s = splits[name]
    for part in ["train", "val", "test"]:
        assert part == "train" or not s.loc[s["split"] == part, "target_key"].isin(
            s.loc[s["split"] == "train", "target_key"]).any()          # cold: no overlap of targets
    rows.append({"dataset": name.upper(),
                 **{f"{p} targets": int((pd.Series(roles[name]) == p).sum()) for p in ["train", "val", "test"]},
                 **{f"{p} pairs": int((s["split"] == p).sum()) for p in ["train", "val", "test"]},
                 "dropped (ligand unseen)": dropped})
pd.DataFrame(rows).set_index("dataset")

## Step 2: Target–target sequence similarity

**Jaccard on 3-mer sets (main method).** Each sequence becomes the set of its overlapping 3-letter fragments. Sets are stored as a 0/1 matrix (one column per fragment that occurs anywhere), so $|A_i \cap A_j|$ is a matrix product and $|A_i \cup A_j| = |A_i| + |A_j| - |A_i \cap A_j|$.

**Supplied Smith–Waterman (reference).** From the DeepDTA files (rows/columns follow `proteins.txt`). DAVIS stores raw scores, normalised to $[0,1]$ by $s'_{ij} = s_{ij} / \sqrt{s_{ii}\, s_{jj}}$ (as in Task 1).

Neither similarity uses any binding score, so there is no label leakage, and both are available for a brand-new target.

In [ ]:
def supplied_similarity(name):
    file = {"davis": "target-target_similarities_WS.txt", "kiba": "kiba_target_sim.txt"}[name]
    S = np.genfromtxt(DATA_DIR / name / file)
    if name == "davis":
        d = np.sqrt(np.diag(S))
        S = S / np.outer(d, d)
    with open(DATA_DIR / name / "proteins.txt") as f:
        ids = list(json.load(f, object_pairs_hook=OrderedDict).keys())
    S = pd.DataFrame(S, index=ids, columns=ids)
    keys = targets[name].index                       # deduplicated targets (representative IDs)
    return S.loc[keys, keys]

def kmer_set(seq, k=3):
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}

def jaccard_similarity(name, k=3):
    keys = targets[name].index
    sets = [kmer_set(s, k) for s in targets[name]["sequence"]]
    vocab = {frag: i for i, frag in enumerate(sorted(set().union(*sets)))}
    B = np.zeros((len(sets), len(vocab)))                 # 1 if the target contains the fragment
    for r, frags in enumerate(sets):
        B[r, [vocab[f] for f in frags]] = 1
    inter = B @ B.T                                       # |A_i intersect A_j|
    size = B.sum(axis=1)
    union = size[:, None] + size[None, :] - inter          # |A_i union A_j|
    return pd.DataFrame(inter / union, index=keys, columns=keys)

MAIN, REF = "Jaccard 3-mer (lecture)", "Smith-Waterman (reference)"
similarities = {name: {MAIN: jaccard_similarity(name), REF: supplied_similarity(name)} for name in pairs}

# how closely related is each cold test target to its nearest training target?
rel = []
for name in pairs:
    tr = [k for k, r in roles[name].items() if r == "train"]
    te = [k for k, r in roles[name].items() if r == "test"]
    for sname, S in similarities[name].items():
        nearest = S.loc[te, tr].max(axis=1)
        rel.append({"dataset": name.upper(), "similarity": sname, "median nearest-train similarity": nearest.median(),
                    "min": nearest.min(), "max": nearest.max()})
pd.DataFrame(rel).round(3)

## Step 3: Models

- **global mean** $\mu$;
- **CF fallback** $\mu + b_u$: what the Task 2 CF predicts for a cold target (lecture baseline without $b_i$). Same score for every cold target of a ligand, so it cannot rank them;
- **sequence neighbours** (formula above) with Jaccard similarity, plus the Smith–Waterman reference.

$\mu$ and $b_u = \bar r_u - \mu$ come from training pairs only. Only positive similarities are used; if a ligand has no positively similar training target, the prediction falls back to $\mu + b_u$.

In [ ]:
def fit(train):
    mu = train["score"].mean()
    b_u = train.groupby("ligand_key")["score"].mean() - mu
    R = train.pivot(index="ligand_key", columns="target_key", values="score")   # training scores only
    return {"mu": mu, "b_u": b_u, "R": R}

def predict(model, df, method, S=None, k=5):
    base = model["mu"] + df["ligand_key"].map(model["b_u"]).values
    if method == "global":
        return np.full(len(df), model["mu"]), np.zeros(len(df), int)
    if method == "fallback":
        return base, np.zeros(len(df), int)

    R = model["R"]
    train_tgts = R.columns
    pred, n_used = base.copy(), np.zeros(len(df), int)
    df = df.reset_index(drop=True)
    for lig, g in df.groupby("ligand_key"):
        scores = R.loc[lig].dropna()                                  # this ligand's training targets
        if scores.empty:
            continue
        dev = scores.values - (model["mu"] + model["b_u"][lig])       # deviations from the ligand mean
        sims = S.loc[g["target_key"].values, scores.index].values     # cold targets x measured train targets
        sims = np.where(sims > 0, sims, 0.0)
        if sims.shape[1] > k:                                         # keep the k most similar per cold target
            cut = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            keep = np.zeros_like(sims, dtype=bool)
            np.put_along_axis(keep, cut, True, axis=1)
            sims = np.where(keep, sims, 0.0)
        wsum = sims.sum(axis=1)
        ok = wsum > 0
        rows = g.index.values
        pred[rows[ok]] += (sims[ok] @ dev) / wsum[ok]
        n_used[rows] = (sims > 0).sum(axis=1)
    return pred, n_used

## Step 4: Metrics

The same metrics as Task 2. Ranking now asks: **among a ligand's cold test targets, is the truly best one ranked first / in the top 5?** This directly tests whether a method can *distinguish between new targets*, which the CF fallback cannot.

In [ ]:
def n_distinct_pairs(x):
    n = len(x)
    _, counts = np.unique(x, return_counts=True)
    return n * (n - 1) / 2 - (counts * (counts - 1) / 2).sum()

def concordance_index(y, yhat):
    A, B = n_distinct_pairs(y), n_distinct_pairs(yhat)
    if A == 0 or B == 0:
        return 0.5
    tau = kendalltau(y, yhat, variant="b").statistic
    return (tau * np.sqrt(B / A) + 1) / 2

def ranking_metrics(df, ks=(1, 5)):
    rows = []
    for lig, g in df.groupby("ligand_key"):
        if g["score"].nunique() < 2:
            continue
        order = g.sort_values("pred", ascending=False, kind="mergesort")
        best = order["score"] == order["score"].max()
        rank = int(np.argmax(best.values)) + 1
        n, n_best = len(g), int(best.sum())
        row = {"rank_of_best": rank}
        for k in ks:
            row[f"hit@{k}"] = float(rank <= k)
            row[f"random hit@{k}"] = 1 - np.prod([(n - n_best - i) / (n - i) for i in range(min(k, n))]) if k < n else 1.0
        rows.append(row)
    return pd.DataFrame(rows)

def evaluate(df, pred, n_used=None):
    scored = df.assign(pred=pred)
    rk = ranking_metrics(scored)
    out = {"MSE": np.mean((scored["score"] - scored["pred"]) ** 2),
           "CI": concordance_index(scored["score"].values, scored["pred"].values),
           "Hit@1": rk["hit@1"].mean(), "Hit@5": rk["hit@5"].mean(),
           "random Hit@1": rk["random hit@1"].mean(), "random Hit@5": rk["random hit@5"].mean(),
           "mean rank of best": rk["rank_of_best"].mean()}
    if n_used is not None:
        out["fallback %"] = 100 * np.mean(n_used == 0)
    return out

## Step 5: Choose $k$ on the cold validation targets

In [ ]:
KS = [1, 2, 3, 5, 10, 20, 40]
models, best_k, search = {}, {}, []
for name, df in splits.items():
    models[name] = fit(df[df["split"] == "train"])
    val = df[df["split"] == "val"]
    best_k[name] = {}
    for sname, S in similarities[name].items():
        scores = {}
        for k in KS:
            pred, _ = predict(models[name], val, "neighbours", S, k)
            scores[k] = np.mean((val["score"].values - pred) ** 2)
            search.append({"dataset": name.upper(), "similarity": sname, "k": k, "val MSE": scores[k]})
        best_k[name][sname] = min(scores, key=scores.get)
    print(name.upper(), "best k:", best_k[name])

search_df = pd.DataFrame(search)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, splits):
    for sname, g in search_df[search_df["dataset"] == name.upper()].groupby("similarity"):
        ax.plot(g["k"], g["val MSE"], marker="o", label=sname)
    fb, _ = predict(models[name], splits[name][splits[name]["split"] == "val"], "fallback")
    v = splits[name][splits[name]["split"] == "val"]["score"].values
    ax.axhline(np.mean((v - fb) ** 2), color="grey", linestyle="--", label="CF fallback (mu + b_u)")
    ax.set_xscale("log")
    ax.set_xlabel("Number of sequence neighbours k")
    ax.set_ylabel("Validation MSE (cold targets)")
    ax.set_title(name.upper())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "cold_target_k_search.png", dpi=200)
plt.show()

## Step 6: Results on the cold test targets (evaluated once)

In [ ]:
results, scored_test = [], {}
for name, df in splits.items():
    test = df[df["split"] == "test"]
    m = models[name]
    for label, method, S, k in [("global mean", "global", None, None),
                                ("CF fallback (mu + b_u)", "fallback", None, None)] + \
                               [(f"neighbours: {s}, k={best_k[name][s]}", "neighbours", Sm, best_k[name][s])
                                for s, Sm in similarities[name].items()]:
        pred, n_used = predict(m, test, method, S, k) if method == "neighbours" else predict(m, test, method)
        results.append({"dataset": name.upper(), "model": label,
                        **evaluate(test, pred, n_used if method == "neighbours" else None)})
        if method == "neighbours":
            scored_test[(name, label)] = test.assign(pred=pred)

results_df = pd.DataFrame(results).set_index(["dataset", "model"])
results_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_df = results_df.reset_index()
plot_df["short"] = plot_df["model"].str.replace("neighbours: ", "").str.replace(r", k=\d+", "", regex=True)
for ax, metric in zip(axes, ["MSE", "CI", "Hit@5"]):
    p = plot_df.pivot(index="dataset", columns="short", values=metric)
    p.plot(kind="bar", ax=ax, rot=0, legend=(metric == "MSE"))
    ax.set_title(f"Cold test targets: {metric}" + (" (lower is better)" if metric == "MSE" else ""))
    ax.set_xlabel("")
if axes[0].get_legend():
    axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "cold_target_results.png", dpi=200)
plt.show()

## Step 7: When does it work? Error by how related a cold target is to the training targets

Each cold test target is grouped (binned, lecture 12.2 slide 6) by its Jaccard similarity to the **nearest training target**. If guilt-by-association is the reason the method works, the gain over the CF fallback should shrink when no close relative exists.

In [ ]:
rows = []
for name, df in splits.items():
    S = similarities[name][MAIN]
    label = f"neighbours: {MAIN}, k={best_k[name][MAIN]}"
    t = scored_test[(name, label)].copy()
    train_tgts = [k for k, r in roles[name].items() if r == "train"]
    nearest = S.loc[:, train_tgts].max(axis=1)
    t["nearest train similarity"] = pd.cut(t["target_key"].map(nearest), [0, 0.15, 0.25, 0.4, 1.0])
    fb, _ = predict(models[name], t, "fallback")
    t["fallback"] = fb
    for b, g in t.groupby("nearest train similarity", observed=True):
        rows.append({"dataset": name.upper(), "nearest-train similarity": str(b),
                     "cold targets": g["target_key"].nunique(), "test pairs": len(g),
                     "neighbours MSE": np.mean((g["score"] - g["pred"]) ** 2),
                     "fallback MSE": np.mean((g["score"] - g["fallback"]) ** 2)})
rel_df = pd.DataFrame(rows)
rel_df["MSE reduction %"] = 100 * (1 - rel_df["neighbours MSE"] / rel_df["fallback MSE"])
rel_df.round(3)